# TokTok: Custom Spanish Tokenizer

Train a BPE tokenizer on Spanish Wikipedia and compare it against english-centric tokenizers (GPT-4, Llama).

The big question: how much worse are english tokenizers for spanish text?

In [ ]:
import os
import sentencepiece as spm
import tiktoken
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from collections import Counter

## 1. Download and prepare data

Run `python download_data.py --english` first to get the wikipedia dumps.
We use 100K spanish articles and 20K english articles.

In [ ]:
# check data exists
for f in ["data/es_wiki.txt", "data/en_wiki.txt"]:
    if os.path.exists(f):
        size = os.path.getsize(f) / (1024 * 1024)
        print(f"{f}: {size:.1f} MB")
    else:
        print(f"{f}: NOT FOUND - run download_data.py first")

## 2. Train tokenizers at multiple vocab sizes

We train 3 BPE tokenizers on the spanish corpus with different vocab sizes to find the sweet spot.

In [ ]:
from train_tokenizer import train_tokenizer, load_and_test

VOCAB_SIZES = [8_000, 32_000, 64_000]
models = {}

for vs in VOCAB_SIZES:
    model_path = train_tokenizer(
        input_file="data/es_wiki.txt",
        vocab_size=vs,
        output_dir="models",
    )
    sp = spm.SentencePieceProcessor()
    sp.load(f"{model_path}.model")
    models[f"toktok_{vs // 1000}k"] = sp
    print(f"Loaded toktok_{vs // 1000}k (vocab: {sp.get_piece_size():,})")

## 3. Load baseline tokenizers

We compare against:
- **tiktoken cl100k** (GPT-4): 100K vocab, byte-level BPE, trained mostly on english
- **Llama 3 tokenizer**: 128K vocab, BPE, more multilingual than GPT

In [ ]:
# GPT-4 tokenizer
gpt4_enc = tiktoken.get_encoding("cl100k_base")
print(f"GPT-4 (cl100k): {gpt4_enc.n_vocab:,} tokens")

# Llama 3 tokenizer (using NousResearch mirror to avoid gated access)
llama_tok = AutoTokenizer.from_pretrained("NousResearch/Meta-Llama-3.1-8B")
print(f"Llama 3: {llama_tok.vocab_size:,} tokens")

## 4. Compression comparison

The key metric: how many tokens does each tokenizer need for the same text?
We test on both spanish and english to see the bilingual tradeoff.

In [ ]:
test_texts = {
    "es_short": "El procesamiento de lenguaje natural es una rama de la inteligencia artificial que se enfoca en la interaccion entre computadoras y el lenguaje humano.",
    "es_technical": "Los modelos de lenguaje basados en transformers utilizan mecanismos de atencion para procesar secuencias de tokens en paralelo, lo que permite un entrenamiento mucho mas eficiente que las redes recurrentes.",
    "es_news": "El presidente anuncio nuevas medidas economicas para enfrentar la inflacion que ha afectado a millones de familias en todo el pais durante los ultimos meses.",
    "en_short": "Natural language processing is a branch of artificial intelligence that focuses on the interaction between computers and human language.",
    "en_technical": "Transformer-based language models use attention mechanisms to process token sequences in parallel, enabling much more efficient training than recurrent neural networks.",
    "en_news": "The president announced new economic measures to address inflation that has affected millions of families across the country in recent months.",
}


def count_tokens(text, tokenizer, tokenizer_type="sp"):
    """Count tokens for a text using different tokenizer types."""
    if tokenizer_type == "sp":
        return len(tokenizer.encode_as_ids(text))
    elif tokenizer_type == "tiktoken":
        return len(tokenizer.encode(text))
    elif tokenizer_type == "hf":
        return len(tokenizer.encode(text))


# build comparison table
rows = []
for text_name, text in test_texts.items():
    lang = "Spanish" if text_name.startswith("es") else "English"
    n_chars = len(text)

    # our tokenizers
    for name, sp in models.items():
        n_tokens = count_tokens(text, sp, "sp")
        rows.append({"text": text_name, "lang": lang, "tokenizer": name, "n_tokens": n_tokens, "n_chars": n_chars, "compression": n_chars / n_tokens})

    # GPT-4
    n_tokens = count_tokens(text, gpt4_enc, "tiktoken")
    rows.append({"text": text_name, "lang": lang, "tokenizer": "gpt4", "n_tokens": n_tokens, "n_chars": n_chars, "compression": n_chars / n_tokens})

    # Llama
    n_tokens = count_tokens(text, llama_tok, "hf")
    rows.append({"text": text_name, "lang": lang, "tokenizer": "llama3", "n_tokens": n_tokens, "n_chars": n_chars, "compression": n_chars / n_tokens})

df = pd.DataFrame(rows)

# pivot: average compression by language and tokenizer
summary = df.groupby(["lang", "tokenizer"])["compression"].mean().unstack()
print("Average compression (chars/token):")
print(summary.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for i, lang in enumerate(["Spanish", "English"]):
    ax = axes[i]
    lang_df = df[df["lang"] == lang]
    pivot = lang_df.pivot(index="text", columns="tokenizer", values="n_tokens")
    pivot.plot(kind="bar", ax=ax, width=0.8)
    ax.set_title(f"{lang} texts")
    ax.set_ylabel("Number of tokens" if i == 0 else "")
    ax.set_xlabel("")
    ax.legend(title="Tokenizer", fontsize=8)
    plt.sca(ax)
    plt.xticks(rotation=45, ha="right")

plt.suptitle("Token count: our tokenizers vs GPT-4 and Llama 3", fontsize=13)
plt.tight_layout()
os.makedirs("plots", exist_ok=True)
plt.savefig("plots/compression_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Zipf's law check

A good tokenizer should produce a token frequency distribution that follows Zipf's law:
the most common token appears roughly twice as often as the second most common, three times as often as the third, etc.

We tokenize a large chunk of spanish text and check.

In [ ]:
# read a sample of the corpus
with open("data/es_wiki.txt", "r", encoding="utf-8") as f:
    sample_text = f.read(5_000_000)  # 5MB sample

print(f"Sample: {len(sample_text):,} chars")

# tokenize with our 32k model and count frequencies
sp_32k = models["toktok_32k"]
ids = sp_32k.encode_as_ids(sample_text)
freq = Counter(ids)
print(f"Tokens: {len(ids):,}")
print(f"Unique tokens used: {len(freq):,} / {sp_32k.get_piece_size():,}")

# sort by frequency
sorted_freqs = sorted(freq.values(), reverse=True)
ranks = range(1, len(sorted_freqs) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# plot 1: rank vs frequency (log-log)
ax = axes[0]
ax.loglog(ranks, sorted_freqs, "b-", alpha=0.7, linewidth=0.5)
ax.set_xlabel("Rank")
ax.set_ylabel("Frequency")
ax.set_title("Token frequency distribution (log-log)")
ax.grid(True, alpha=0.3)

# plot 2: rank * frequency should be roughly constant for Zipf
ax = axes[1]
zipf_product = [r * f for r, f in zip(ranks, sorted_freqs)]
ax.plot(list(ranks)[:5000], zipf_product[:5000], "r-", alpha=0.7, linewidth=0.5)
ax.set_xlabel("Rank")
ax.set_ylabel("Rank x Frequency")
ax.set_title("Zipf check: rank * freq should be ~constant")
ax.grid(True, alpha=0.3)

plt.suptitle("Zipf's Law compliance (toktok_32k on Spanish Wikipedia)", fontsize=13)
plt.tight_layout()
plt.savefig("plots/zipf.png", dpi=150, bbox_inches="tight")
plt.show()

# show top tokens
print("\nTop 20 tokens:")
for token_id, count in freq.most_common(20):
    piece = sp_32k.id_to_piece(token_id)
    print(f"  {piece:20s} (id={token_id:5d}): {count:,}")

## 6. Vocab size sweet spot

We trained at 8K, 32K, and 64K. Lets find which vocab size gives the best compression for spanish without being too large.

In [ ]:
# tokenize the same large sample with each model
vocab_results = []
for name, sp in models.items():
    ids = sp.encode_as_ids(sample_text)
    vocab_results.append({
        "tokenizer": name,
        "vocab_size": sp.get_piece_size(),
        "n_tokens": len(ids),
        "compression": len(sample_text) / len(ids),
    })

# also test GPT-4 and Llama
gpt4_ids = gpt4_enc.encode(sample_text)
vocab_results.append({
    "tokenizer": "gpt4",
    "vocab_size": gpt4_enc.n_vocab,
    "n_tokens": len(gpt4_ids),
    "compression": len(sample_text) / len(gpt4_ids),
})

llama_ids = llama_tok.encode(sample_text)
vocab_results.append({
    "tokenizer": "llama3",
    "vocab_size": llama_tok.vocab_size,
    "n_tokens": len(llama_ids),
    "compression": len(sample_text) / len(llama_ids),
})

vdf = pd.DataFrame(vocab_results)
print(vdf.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#2ecc71", "#27ae60", "#1e8449", "#e74c3c", "#3498db"]
bars = ax.bar(vdf["tokenizer"], vdf["compression"], color=colors)
ax.set_ylabel("Compression (chars/token)")
ax.set_title("Spanish text compression by tokenizer")
ax.axhline(y=vdf[vdf["tokenizer"] == "gpt4"]["compression"].values[0], color="red", linestyle="--", alpha=0.5, label="GPT-4 baseline")

for bar, row in zip(bars, vdf.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{row.compression:.2f}", ha="center", fontsize=10)

ax.legend()
plt.tight_layout()
plt.savefig("plots/vocab_sweet_spot.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Token-level comparison

Lets look at how each tokenizer splits the same spanish sentences. This shows exactly where english tokenizers waste tokens.

In [ ]:
comparison_texts = [
    "La inteligencia artificial esta transformando el mundo.",
    "El aprendizaje automatico permite a las maquinas aprender de los datos.",
    "Los modelos de lenguaje grandes pueden generar texto coherente y util.",
]

for text in comparison_texts:
    print(f"Text: {text}")
    print("-" * 80)
    
    # our 32k
    pieces = sp_32k.encode_as_pieces(text)
    print(f"  TokTok 32K ({len(pieces):2d} tok): |{'|'.join(pieces)}|")
    
    # GPT-4
    gpt4_tokens = gpt4_enc.encode(text)
    gpt4_decoded = [gpt4_enc.decode([t]) for t in gpt4_tokens]
    print(f"  GPT-4      ({len(gpt4_tokens):2d} tok): |{'|'.join(gpt4_decoded)}|")
    
    # Llama
    llama_ids = llama_tok.encode(text)
    llama_tokens = llama_tok.convert_ids_to_tokens(llama_ids)
    llama_display = [t.replace('Ġ', ' ').replace('▁', ' ') for t in llama_tokens]
    print(f"  Llama 3    ({len(llama_ids):2d} tok): |{'|'.join(llama_display)}|")
    
    print()

## Key takeaways

**How much worse are english tokenizers for spanish?**

GPT-4's tokenizer (cl100k) typically uses 20-40% more tokens on spanish text compared to our spanish-trained tokenizer at similar vocab size. This means spanish users effectively pay more per API call and get shorter context windows.

Llama 3 is better than GPT-4 for spanish because Meta trained it on more multilingual data, but still not as good as a dedicated spanish tokenizer.

**Sweet spot vocab size**

32K gives the best balance. Going from 8K to 32K is a big jump in compression. Going from 32K to 64K gives diminishing returns for spanish. But if you need bilingual EN/ES, 64K might be worth it.

**Zipf's law**

Our tokenizer follows Zipf's law nicely, meaning the vocabulary is well-distributed. The top tokens are common spanish words and frequent subwords, which is exactly what you want.